# Anti-Money Laundering (AML) Graph Analytics

### Fraud Detection Aml Analytics — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), and basic understanding of fraud detection and anti-money laundering terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Describe the information-extraction task and its relevance to fraud detection and anti-money laundering.
- Implement a rule-based extraction pipeline and evaluate accuracy on synthetic examples.
- Assess limitations of rule-based extraction and when ML-based approaches would be more appropriate.

**Estimated time:** 35–45 minutes

**Collection context:** Fraud detection, AML compliance, anomaly detection, and financial crime analytics in banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Money laundering often involves "layering"—moving money through multiple accounts to hide its source. Standard tabular ML might miss these patterns, but Graph Analytics excels at finding connections.

**Input data used:** Account IDs and transaction history (Sender -> Receiver).

**What we detect:** Suspicious "Circular Transfers" and "High-Centrality" nodes that act as hubs for illicit funds.

**Method used:** NetworkX library for graph construction and cycle detection.

**Learning goal:** Understand how financial data can be treated as a network to uncover complex fraud rings.

**Primary stakeholders:** fraud analysts, compliance officers, risk managers, and financial crime investigators.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Create Synthetic Financial Network

We generate 50 accounts. Most transactions are random, but we will intentionally inject a "Money Laundering Cycle" (A -> B -> C -> A).

In [ ]:
G = nx.DiGraph()

# Create 50 accounts
accounts = [f"ACC_{i:03d}" for i in range(1, 51)]
G.add_nodes_from(accounts)

# Add 100 normal transactions
for _ in range(100):
    u, v = RNG.choice(accounts, 2, replace=False)
    G.add_edge(u, v, weight=RNG.integers(100, 5000))

# INJECT A SUSPICIOUS CYCLE (The Red Flag)
laundry_ring = ["ACC_010", "ACC_025", "ACC_042", "ACC_010"]
for i in range(len(laundry_ring)-1):
    G.add_edge(laundry_ring[i], laundry_ring[i+1], weight=10000, label="Suspicious")

print(f"Network created with {G.number_of_nodes()} accounts and {G.number_of_edges()} transactions.")

## Step 4 — Exploratory Data Analysis

Visualization

In a small network, we can sometimes spot patterns visually. Let's highlight high-value transfers.

In [ ]:
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=0.5, seed=42)

# Draw normal nodes and edges
nx.draw_networkx_nodes(G, pos, node_size=500, node_color='lightblue')
nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=10, edge_color='gray', alpha=0.3)

# Highlight the suspicious cycle
cycle_edges = [("ACC_010", "ACC_025"), ("ACC_025", "ACC_042"), ("ACC_042", "ACC_010")]
nx.draw_networkx_edges(G, pos, edgelist=cycle_edges, edge_color='#E76F51', width=2)

plt.title("Financial Transaction Network (Red = Potential Money Laundering Cycle)")
plt.axis('off')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Visualisation titled "Financial Transaction Network (Red = Potential Money Laundering Cycle)". The chart highlights key patterns that inform the modelling approach.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

For this workflow, the data prepared in Step 3 is used directly as input features.

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: manual extraction (no automation) ---
print("Baseline: without automation, each document must be read and keyed manually.")
print("Any automated approach that achieves >80% field accuracy saves significant time.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
print("Searching for suspicious circular transfer patterns...")
cycles = list(nx.simple_cycles(G))

# Filter for cycles of length 3 or more (typical layering)
layering_patterns = [c for c in cycles if len(c) >= 3]

for i, cycle in enumerate(layering_patterns):
    print(f"Potential Laundering Ring {i+1}: {' -> '.join(cycle)} -> {cycle[0]}")

In [ ]:
pagerank = nx.pagerank(G, weight='weight')
top_accounts = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:5]

print("Top 5 'Hub' Accounts (Potential Shell Companies):")
for acc, score in top_accounts:
    print(f"{acc}: Score {score:.4f}")

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary of Findings:**
1. **Cycle Detection:** We found the injected cycle `ACC_010 -> ACC_025 -> ACC_042`. In banking, this is often a sign of "Round Tripping" to inflate transaction volumes or hide fund origins.
2. **Centrality:** Accounts with very high PageRank but no clear business purpose might be "mule" accounts or shell companies.

**Why Graph Analytics works here:** Unlike traditional models that look at single transactions, Graph Analytics looks at the *relationships* between entities, which is where financial crimes are often hidden.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: process a new document ---
print("Operational demonstration — field extraction:\n")
new_doc = "Invoice #2055. Vendor: Wayne Enterprises. Date: 2024-06-15. Total: $8750.00"
result = extract_info(new_doc)
print(f"  Raw: {new_doc}")
print(f"  Vendor: {result[0]}  Date: {result[1]}  Amount: ${result[2]:,.2f}")
print("\nExtracted fields would auto-populate the accounts-payable system.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end fraud detection and anti-money laundering workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- False positives can freeze legitimate accounts, causing financial hardship and customer distrust.
- Fraud models risk encoding biases against specific demographic groups or geographic regions.
- Transaction monitoring must balance fraud prevention with customer privacy rights.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in fraud detection and anti-money laundering?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.